# MLflow Tracing para LLMs con LangChain y Google Gemini

En este notebook exploraremos las nuevas capacidades de **MLflow Tracing** para el seguimiento y observabilidad de modelos de lenguaje (LLMs). 

## ¿Qué es MLflow Tracing?

MLflow Tracing es un sistema de observabilidad que captura automáticamente:
- **Prompts y respuestas** enviados/recibidos del modelo
- **Latencias** de cada llamada
- **Tokens utilizados** (input, output, total)
- **Metadatos** (temperatura, modelo, etc.)
- **Excepciones** si ocurren errores

## Integraciones Soportadas

MLflow soporta **40+ integraciones** incluyendo:
- **Frameworks de Agentes:** LangChain, LangGraph, DSPy, CrewAI, LlamaIndex
- **Proveedores de Modelos:** OpenAI, Anthropic, Google Gemini, Mistral, Ollama

## ¿Qué aprenderemos?

1. Usar `mlflow.gemini.autolog()` para tracing directo con Google GenAI SDK
2. Usar `mlflow.langchain.autolog()` para tracing con LangChain
3. Visualizar trazas en la UI de MLflow
4. Comparar diferentes configuraciones de prompts

## 1. Instalación de Dependencias

Instalamos las librerías necesarias. Nota: Se requiere MLflow >= 2.14.0 para tracing con LangChain.

In [1]:
# Instalamos las dependencias necesarias
!pip install "mlflow>=3.10.0" langchain langchain-google-genai google-generativeai python-dotenv

## 2. Configuración Inicial

In [ ]:
import os
import mlflow
from dotenv import load_dotenv

# Cargamos variables de entorno
load_dotenv()

# ========================================
# IMPORTANTE: Configurar el Tracking URI
# ========================================
# Esto conecta con tu servidor MLflow desplegado
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")

# Configuramos la API Key de Google
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("Configura GOOGLE_API_KEY en tu archivo .env")

print("API Key configurada correctamente")

---
# PARTE 1: MLflow Tracing con Google GenAI SDK Directo

Primero veremos cómo usar `mlflow.gemini.autolog()` para tracing automático con el SDK nativo de Google.

In [ ]:
import google.generativeai as genai

# Habilitamos el auto-tracing para Gemini
mlflow.gemini.autolog()

# Configuramos el experimento
mlflow.set_experiment("Gemini_Direct_Tracing")

# Configuramos el SDK de Google
genai.configure(api_key=GOOGLE_API_KEY)

print("MLflow Gemini autolog habilitado")
print(f"Experimento: Gemini_Direct_Tracing")

In [ ]:
# Creamos el modelo
model = genai.GenerativeModel("gemini-2.5-flash")

# Realizamos una llamada simple - MLflow capturará automáticamente la traza
response = model.generate_content("¿Qué es MLflow y para qué se utiliza? Responde en 2-3 oraciones.")

print("Respuesta:")
print(response.text)

In [ ]:
# Ejemplo con chat multi-turno
chat = model.start_chat(history=[])

# Primera pregunta
response1 = chat.send_message("Explica qué es Docker en una oración.")
print("Turno 1:", response1.text)

# Segunda pregunta (contexto mantenido)
response2 = chat.send_message("¿Y Kubernetes? También en una oración.")
print("Turno 2:", response2.text)

---
# PARTE 2: MLflow Tracing con LangChain

Ahora veremos cómo usar `mlflow.langchain.autolog()` para tracing automático con LangChain.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

# Habilitamos el auto-tracing para LangChain
mlflow.langchain.autolog()

# Configuramos el experimento
mlflow.set_experiment("LangChain_Gemini_Tracing")

print("MLflow LangChain autolog habilitado")
print(f"Experimento: LangChain_Gemini_Tracing")

In [ ]:
# Inicializamos el modelo de LangChain
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.7,
    max_tokens=1024
)

print("Modelo LangChain inicializado")

### 2.1 Llamada Simple con Mensajes

In [ ]:
# Llamada simple - la traza se captura automáticamente
messages = [
    SystemMessage(content="Eres un experto en MLOps."),
    HumanMessage(content="¿Qué es el tracking de experimentos en ML?")
]

response = llm.invoke(messages)

print("Respuesta:")
print(response.content)

### 2.2 Usando Chains con PromptTemplate

Las chains de LangChain son trazadas automáticamente, capturando cada paso.

In [ ]:
# Creamos un prompt template parametrizado
template = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en {tema}. Responde de forma {estilo}."),
    ("human", "{pregunta}")
])

# Creamos la chain: prompt -> llm -> parser
chain = template | llm | StrOutputParser()

print("Chain creada: PromptTemplate -> LLM -> StrOutputParser")

In [ ]:
# Ejecutamos la chain - MLflow traza cada componente
result = chain.invoke({
    "tema": "Kubernetes",
    "estilo": "concisa y técnica",
    "pregunta": "¿Cuáles son los componentes principales de un cluster?"
})

print("Respuesta de la Chain:")
print(result)

### 2.3 Comparación de Diferentes Prompts

Ejecutamos la misma pregunta con diferentes system prompts para comparar en MLflow UI.

In [ ]:
# Diferentes versiones de prompts para comparar
prompt_versions = [
    {
        "name": "basico",
        "system": "Eres un asistente útil.",
        "user": "¿Qué es Docker?"
    },
    {
        "name": "experto",
        "system": "Eres un experto en DevOps con 10 años de experiencia. Responde de forma técnica pero accesible.",
        "user": "¿Qué es Docker?"
    },
    {
        "name": "estructurado",
        "system": "Eres un experto en DevOps. Responde usando: 1) Definición, 2) Casos de uso, 3) Ventajas.",
        "user": "¿Qué es Docker?"
    }
]

for version in prompt_versions:
    print(f"\n{'='*50}")
    print(f"Versión: {version['name']}")
    print(f"{'='*50}")
    
    messages = [
        SystemMessage(content=version["system"]),
        HumanMessage(content=version["user"])
    ]
    
    response = llm.invoke(messages)
    print(response.content[:300] + "...")

---
# PARTE 3: Logging Manual Adicional

Además del tracing automático, podemos añadir métricas y artifacts manualmente.

In [ ]:
import time

def experiment_with_run(run_name, system_prompt, user_prompt, temperature=0.7):
    """
    Ejecuta una llamada al LLM dentro de un MLflow run para logging adicional.
    """
    with mlflow.start_run(run_name=run_name):
        # Loggeamos parámetros
        mlflow.log_param("model", "gemini-1.5-flash")
        mlflow.log_param("temperature", temperature)
        mlflow.log_param("system_prompt", system_prompt[:200])
        mlflow.log_param("user_prompt", user_prompt[:200])
        
        # Creamos modelo con temperatura específica
        llm_temp = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash-lite",
            google_api_key=GOOGLE_API_KEY,
            temperature=temperature
        )
        
        # Medimos latencia
        start = time.time()
        
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        response = llm_temp.invoke(messages)
        
        latency = time.time() - start
        
        # Loggeamos métricas
        mlflow.log_metric("latency_seconds", latency)
        mlflow.log_metric("response_length", len(response.content))
        
        # Guardamos la respuesta como artifact
        mlflow.log_text(response.content, "response.txt")
        
        # Guardamos datos estructurados
        mlflow.log_dict({
            "system_prompt": system_prompt,
            "user_prompt": user_prompt,
            "response": response.content,
            "latency": latency
        }, "interaction.json")
        
        print(f"Run: {run_name}")
        print(f"Latencia: {latency:.2f}s")
        print(f"Respuesta: {response.content[:150]}...")
        
        return response.content

In [ ]:
# Ejecutamos experimentos con diferentes temperaturas
mlflow.set_experiment("Temperature_Comparison")

temperatures = [0.0, 0.5, 1.0]
prompt = "Escribe una descripción creativa de qué es la Inteligencia Artificial."

for temp in temperatures:
    print(f"\n{'='*50}")
    experiment_with_run(
        run_name=f"temp_{temp}",
        system_prompt="Eres un escritor creativo.",
        user_prompt=prompt,
        temperature=temp
    )

---
# PARTE 4: Tracing Manual con Decorador

Podemos usar `@mlflow.trace` para crear spans personalizados.

In [ ]:
@mlflow.trace
def generate_summary(text: str, max_words: int = 50) -> str:
    """
    Genera un resumen del texto.
    MLflow captura inputs, outputs y latencia automáticamente.
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"Resume el siguiente texto en máximo {max_words} palabras."),
        ("human", "{text}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"text": text})


@mlflow.trace
def analyze_and_summarize(text: str) -> dict:
    """
    Pipeline que analiza y resume texto.
    Cada función decorada crea un span en la traza.
    """
    summary = generate_summary(text, max_words=30)
    
    return {
        "original_length": len(text),
        "summary": summary,
        "summary_length": len(summary)
    }

In [ ]:
mlflow.set_experiment("Custom_Tracing")

texto = """
MLflow es una plataforma de código abierto para gestionar el ciclo de vida de ML.
Fue desarrollada por Databricks y proporciona cuatro componentes: Tracking para 
registrar experimentos, Projects para empaquetar código, Models para gestionar 
modelos, y Model Registry para versionar modelos en producción. Recientemente 
añadió soporte para LLMs con tracing automático.
"""

result = analyze_and_summarize(texto)
print(f"Texto original: {result['original_length']} caracteres")
print(f"Resumen: {result['summary']}")

In [ ]:
@mlflow.trace
def prompt_tracint(llm): 
    prompt_versions = [
        {
            "name": "basico",
            "system": "Eres un asistente útil.",
            "user": "¿Qué es Docker?"
        },
        {
            "name": "experto",
            "system": "Eres un experto en DevOps con 10 años de experiencia. Responde de forma técnica pero accesible.",
            "user": "¿Qué es Docker?"
        },
        {
            "name": "estructurado",
            "system": "Eres un experto en DevOps. Responde usando: 1) Definición, 2) Casos de uso, 3) Ventajas.",
            "user": "¿Qué es Docker?"
        }
    ]

    for version in prompt_versions:
        print(f"\n{'='*50}")
        print(f"Versión: {version['name']}")
        print(f"{'='*50}")
        
        messages = [
            SystemMessage(content=version["system"]),
            HumanMessage(content=version["user"])
        ]
        
        response = llm.invoke(messages)
        print(response.content[:300] + "...")

---
# PARTE 5: Evaluación de Respuestas

In [ ]:
import pandas as pd

# Dataset de evaluación
eval_data = pd.DataFrame({
    "question": [
        "¿Qué es un contenedor en Docker?",
        "¿Para qué sirve Kubernetes?",
        "¿Qué es CI/CD?"
    ],
    "expected_keywords": [
        ["software", "dependencias", "aislado"],
        ["orquestación", "contenedores", "escalado"],
        ["integración", "continua", "despliegue"]
    ]
})

print(eval_data)

In [ ]:
mlflow.set_experiment("LLM_Evaluation")

with mlflow.start_run(run_name="evaluation_run"):
    results = []
    
    for idx, row in eval_data.iterrows():
        start = time.time()
        
        messages = [
            SystemMessage(content="Responde de forma concisa y técnica."),
            HumanMessage(content=row["question"])
        ]
        response = llm.invoke(messages)
        
        latency = time.time() - start
        
        # Verificamos keywords presentes
        response_lower = response.content.lower()
        keywords_found = sum(1 for kw in row["expected_keywords"] if kw in response_lower)
        keyword_score = keywords_found / len(row["expected_keywords"])
        
        results.append({
            "question": row["question"],
            "response": response.content,
            "latency": latency,
            "keyword_score": keyword_score
        })
        
        mlflow.log_metric(f"latency_q{idx}", latency)
        mlflow.log_metric(f"keyword_score_q{idx}", keyword_score)
    
    # Métricas agregadas
    results_df = pd.DataFrame(results)
    mlflow.log_metric("avg_latency", results_df["latency"].mean())
    mlflow.log_metric("avg_keyword_score", results_df["keyword_score"].mean())
    
    # Guardamos resultados
    mlflow.log_table(results_df, "evaluation_results.json")
    
    print("Resultados de evaluación:")
    print(results_df[["question", "latency", "keyword_score"]])